In [ ]:
import torch
import math
import torch.nn as nn

"""
there are many reasons for using the nn.Parameters
i)we can't be able to use the model.parameters() when we did try to use the backpropagation in the optimizer
ii)we can't move our model weights to cuda unless explicitly written
iii)it won't save the model weights with the command


But honestly our main aim is not to oppose torch functions and modules
we need to use them but we need to write the code that can be understood in a way we can upgrade ourself from the custom implementations to torch upgrades

so we need to think of the basic example for this to understand 
lets consider a famous example of the handwritten digit identification process 
there is a image of a digit is given and then our role is to find which digit it is by our computer
in this scenario the example considers the gray scale image 
now consider a (30px,30px) image so these are the length and then breadth
now there is a image we can see in that 900 boxes but the computer can see the 0's and 1's 
how can it distinguish that image,its simple 
on which the boxes the number was there,because the entire number itself cam't be in a single box right,so it would be distributed into the different boxes 
and ,we can easily understand that,now you can check the corners so ,we can say ,there is no number in that edges so that boxes is now considered as the 0
and then boxes with the number's presence is considered the value in between the 0 and  1 and depending on the intensity of the number inside that unit

so these are the neurons,they are basically the values between the 0 and 1 

and now we need to come up with the solution of the weights 
now before coming to the weights we need to know about the 

in_features = 900, these are the 900 neurons
out_features = 10 (we only need 10,why?because in this case,our goal is to reduce the 900 neurons into the 10 neurons which say the digit is either 0 ,1 .... 9 ,any one of them)

bias -> this is the threshold required to give a small push to the neuron,because sometimes they need to fire up in a way that,the answer might be the digit 8
but the neuron that is a bit low on the activation energy to trigger,so we apply this bias to use as a threshold and then it wll trigger the digit 8 as the answer from the model

Positive bias: Lowers the barrier so the neuron fires even with weak input.
Negative bias: Raises the barrier so the neuron stays quiet unless the evidence is overwhelming.
Even when inputs are dead silent, the bias represents the baseline prior probability or default state of the neuron.

the intresting part is the weights
so the weights are the matrix which are in the shape of the (in_features,out_features) or (out_features,in_features) ->depends on your wish as you can transpose at the forward method
what exactly these weights are ???

input neurons(in_features) are gonna effect the output neurons (out_features) because we are reducing them to the size of the output neurons
so, each input neuron is connected to the each output neuron
(but why ? the reason is,because of that we can know how far does it effects the other numbers as the output,as the inut nueron 1 might contribute more to the digit 1 and then very very less to the digit 9 
in this case,these all 900 neurons will be taken into the charge and the final neuron is activated ,it is that digit )
so as you can see,we can say

Each of the 900 input neurons connects to ALL 10 output neurons.
The weight matrix (10, 900) stores these connections: row i contains 900 weights showing how much EACH input pixel influences output neuron i (which represents digit i)


"""

"""
what happens when you don't use the nn.Parameter
then you need to manually wriite the new methods

self.weights_grad = None  
self.bias_grad = None 
                
"""


"""
you can use the requires_grad=True where the torch actually calculates you the gradients
so u can use the pytorch's autograd to calculate the backprop 
so u can use the simple p.grad for the parameters

u can use the nn.Parameter if you want,after inheriting from the nn.Module
1)autonatic parameter registration
2)no need of writing the requires_grad=True
3)device migration
4)saving the states of the model


when you create the torch.randn() pytorch creates the data attribyure related to that 
so we can use the p.data 
"""

"""
use of the kaim scaling

using the torch randn method we can initialize the mean -> 0 and variance -> 1
and with the kaiming we can initialize the mean -> 0 var - > (std)^2

imagine with no bias
E[Y]=E[XW]=E[X]⋅E[W]
y = xw
if we did imagine the x and w are the independent random variables(as the data doesn't know how we initialized the weights)

The Expected Value (Mean) of a product of two independent variables is:
E[Y]=E[X⋅W]=E[X]⋅E[W]

as the torch.randn makes the mean 0 to make them symmetical around zero,E[W] = 0
and E[Y] = 0

Because the mean of Y is always 0, the mean of the gradient E[∂L/∂W] also remains 0 throughout training (assuming no bias).
This is great because it means your updates ΔWΔW are centered around zero, preventing all your weights from drifting permanently positive or negative.
If E[W]≠0E[W]=0, your updates would have a systematic bias, causing a "drift" effect where all neurons learn the same thing.




Y = X1W1+X2W2+X3W3+....
vari(Y) = sum of the individual variances

var(Xi*Wi)=Var(Xi)*Var(Wi)

so if we did assume all the inputs and weights have the same variance and then 
Var(Y)=i=1∑D*(σX^2*σW^2)=D*σX^2*σW^2


Var(Y)=D*Var(X)*Var(W)

If you use torch.randn (Std = 1), then Var(W)=1. Var(Y)=D*Var(X)

If you use Kaiming (Std = 2/D), then Var(W)=2/D. Var(Y)=D⋅Var(X)⋅(2/D)=2⋅Var(X)


"""

class LinearLayer:

    def __init__(self,in_features,out_features):
        
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.bias = torch.zeros(out_features)

        # we can use the requires_grad=True to avoid this 
        self.weights_grad = None  
        self.bias_grad = None 
        
        #weights are indeed in the reverse shape here, (out,in), because we did multiply with the x-> (B,in_features) @ weights->(out_features,in_features).T 
        # so we apply the transpose ,so we usually assign the weights in the tranpose order
        # there is no mandatory to take the weights only in this way,its all upto you

    def forward(self,x):
            return x @ self.weights.T+self.bias


B = 2
in_features= 32
out_features = 16

linear = LinearLayer(in_features,out_features)

x = torch.randn(B,in_features)
y = linear.forward(x)
y.shape

torch.Size([2, 16])